# Apples to Apples: Token-Level vs Byte-Level

Same GPU. Same data. Same optimizer. Same steps.
The only difference: 1024-token vocab vs 260-byte vocab.

**Runtime > H100 or A100 > Run all**

In [ ]:
# ============================================================
# SETUP
# ============================================================
!pip install -q torch numpy huggingface-hub sentencepiece

import os, shutil, math, time, numpy as np
from pathlib import Path
from huggingface_hub import hf_hub_download
import sentencepiece as spm
import torch
import torch.nn.functional as F
from torch import nn

assert torch.cuda.is_available(), 'No GPU!'
device = torch.device('cuda')
gpu = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu} ({mem:.0f}GB)')

In [ ]:
# ============================================================
# DATA — both token-level and byte-level from same 2 shards
# ============================================================
os.makedirs('data/bytes', exist_ok=True)

def dl(fn, sub, dst):
    if Path(dst).exists(): return
    print(f'  Downloading {fn}...')
    c = str(Path(hf_hub_download(repo_id='willdepueoai/parameter-golf',
        filename=fn, subfolder=sub, repo_type='dataset')).resolve())
    try: os.link(c, dst)
    except OSError: shutil.copy2(c, dst)

dl('fineweb_train_000000.bin', 'datasets/datasets/fineweb10B_sp1024', 'data/train0.bin')
dl('fineweb_train_000001.bin', 'datasets/datasets/fineweb10B_sp1024', 'data/train1.bin')
dl('fineweb_val_000000.bin', 'datasets/datasets/fineweb10B_sp1024', 'data/val.bin')
dl('fineweb_1024_bpe.model', 'datasets/tokenizers', 'data/tok.model')

def load_shard(p):
    h = np.fromfile(p, dtype='<i4', count=256)
    return np.fromfile(p, dtype='<u2', count=int(h[2]), offset=256*4)

def write_shard(p, d):
    h = np.zeros(256, dtype='<i4')
    h[0] = 20240520; h[1] = 1; h[2] = len(d)
    with open(p, 'wb') as f:
        f.write(h.tobytes())
        f.write(d.astype('<u2').tobytes())

# Token-level data (direct from shards)
token_train = np.concatenate([
    load_shard('data/train0.bin').astype(np.int64),
    load_shard('data/train1.bin').astype(np.int64),
])
token_val = load_shard('data/val.bin').astype(np.int64)

# Byte-level data (converted)
sp = spm.SentencePieceProcessor(model_file='data/tok.model')
for src, dst in [('data/train0.bin','data/bytes/train0.bin'),
                 ('data/train1.bin','data/bytes/train1.bin'),
                 ('data/val.bin','data/bytes/val.bin')]:
    if Path(dst).exists(): continue
    print(f'  Converting {src}...')
    tok = load_shard(src)
    bs = [np.frombuffer(sp.decode(tok[j:j+100000].tolist()).encode('utf-8'), dtype=np.uint8)
          for j in range(0, len(tok), 100000)]
    write_shard(dst, np.concatenate(bs))

def load_bytes(p):
    h = np.fromfile(p, dtype='<i4', count=256)
    return np.fromfile(p, dtype='<u2', count=int(h[2]), offset=256*4).astype(np.int64)

byte_train = np.concatenate([load_bytes('data/bytes/train0.bin'), load_bytes('data/bytes/train1.bin')])
byte_val = load_bytes('data/bytes/val.bin')

print(f'Token train: {len(token_train):,} tokens | Token val: {len(token_val):,} tokens')
print(f'Byte train:  {len(byte_train):,} bytes  | Byte val:  {len(byte_val):,} bytes')
print(f'Ratio: {len(byte_train)/len(token_train):.2f} bytes per token')

In [ ]:
# ============================================================
# SHARED BUILDING BLOCKS — identical architecture, different vocab
# ============================================================

class Rotary(nn.Module):
    def __init__(self, dim, base=10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
        self.register_buffer('inv_freq', inv_freq, persistent=False)
        self._cache = None
    def forward(self, seq_len, device, dtype):
        if self._cache is None or self._cache[0] != seq_len:
            t = torch.arange(seq_len, device=device, dtype=self.inv_freq.dtype)
            freqs = torch.outer(t, self.inv_freq.to(device))
            self._cache = (seq_len, freqs.cos()[None,None,:,:], freqs.sin()[None,None,:,:])
        return self._cache[1].to(dtype), self._cache[2].to(dtype)

def apply_rope(x, cos, sin):
    h = x.size(-1) // 2
    x1, x2 = x[..., :h], x[..., h:]
    return torch.cat((x1*cos + x2*sin, x1*(-sin) + x2*cos), dim=-1)

class Attn(nn.Module):
    def __init__(self, dim, nh, nkv, qk_gain):
        super().__init__()
        self.nh, self.nkv, self.hd = nh, nkv, dim // nh
        kv = nkv * self.hd
        self.cq = nn.Linear(dim, dim, bias=False)
        self.ck = nn.Linear(dim, kv, bias=False)
        self.cv = nn.Linear(dim, kv, bias=False)
        self.proj = nn.Linear(dim, dim, bias=False)
        nn.init.zeros_(self.proj.weight)
        self.qg = nn.Parameter(torch.full((nh,), qk_gain))
        self.rot = Rotary(self.hd)
    def forward(self, x):
        B, S, D = x.shape
        q = self.cq(x).reshape(B,S,self.nh,self.hd).transpose(1,2)
        k = self.ck(x).reshape(B,S,self.nkv,self.hd).transpose(1,2)
        v = self.cv(x).reshape(B,S,self.nkv,self.hd).transpose(1,2)
        q, k = F.rms_norm(q,(self.hd,)), F.rms_norm(k,(self.hd,))
        cos, sin = self.rot(S, x.device, q.dtype)
        q, k = apply_rope(q,cos,sin), apply_rope(k,cos,sin)
        q = q * self.qg.to(q.dtype)[None,:,None,None]
        y = F.scaled_dot_product_attention(q,k,v, is_causal=True, enable_gqa=(self.nkv!=self.nh))
        return self.proj(y.transpose(1,2).contiguous().reshape(B,S,D))

class MLP(nn.Module):
    def __init__(self, dim, mult):
        super().__init__()
        self.fc = nn.Linear(dim, dim*mult, bias=False)
        self.proj = nn.Linear(dim*mult, dim, bias=False)
        nn.init.zeros_(self.proj.weight)
    def forward(self, x):
        return self.proj(F.leaky_relu(self.fc(x), 0.5).square())

class Block(nn.Module):
    def __init__(self, dim, nh, nkv, mult, qk_gain):
        super().__init__()
        self.attn = Attn(dim, nh, nkv, qk_gain)
        self.mlp = MLP(dim, mult)
        self.as_ = nn.Parameter(torch.ones(dim))
        self.ms = nn.Parameter(torch.ones(dim))
        self.rm = nn.Parameter(torch.stack((torch.ones(dim), torch.zeros(dim))))
    def forward(self, x, x0):
        m = self.rm.to(x.dtype)
        x = m[0][None,None,:]*x + m[1][None,None,:]*x0
        x = x + self.as_.to(x.dtype)[None,None,:] * self.attn(F.rms_norm(x,(x.size(-1),)))
        x = x + self.ms.to(x.dtype)[None,None,:] * self.mlp(F.rms_norm(x,(x.size(-1),)))
        return x

class LM(nn.Module):
    """Standard autoregressive LM. Works for both token and byte level."""
    def __init__(self, vocab_size, dim, num_layers, nh, nkv, mlp_mult, softcap=30.0):
        super().__init__()
        self.dim = dim
        self.sc = softcap
        self.vocab = vocab_size
        self.embed = nn.Embedding(vocab_size, dim)
        nn.init.normal_(self.embed.weight, std=0.005)
        ne = num_layers // 2
        nd = num_layers - ne
        self.ne, self.nd = ne, nd
        self.sw = nn.Parameter(torch.ones(min(ne, nd), dim))
        self.blocks = nn.ModuleList([Block(dim, nh, nkv, mlp_mult, 1.5) for _ in range(num_layers)])

    def forward(self, ids, tgt):
        x = F.rms_norm(self.embed(ids), (self.dim,))
        x0 = x
        skips = []
        for i in range(self.ne):
            x = self.blocks[i](x, x0)
            skips.append(x)
        for i in range(self.nd):
            if skips:
                x = x + self.sw[i].to(x.dtype)[None,None,:] * skips.pop()
            x = self.blocks[self.ne + i](x, x0)
        x = F.rms_norm(x, (x.size(-1),))
        logits = F.linear(x, self.embed.weight)  # tied embeddings
        logits = self.sc * torch.tanh(logits / self.sc)
        return F.cross_entropy(logits.float().reshape(-1, self.vocab), tgt.reshape(-1))

n_params = lambda m: sum(p.numel() for p in m.parameters())
print('Model defined')

In [ ]:
# ============================================================
# TRAINING HARNESS
# ============================================================

STEPS = 7000
LR = 3e-4
LOG_EVERY = 1000
EMA_DECAY = 0.997

def train_and_val(name, model, train_data, val_data, seq_len, batch_bytes, is_byte_level):
    model = model.to(device).bfloat16()
    p = n_params(model)
    print(f'\n{"="*60}')
    print(f'{name}: {p:,} params ({p/1e6:.1f}M) | seq={seq_len}')
    print(f'{"="*60}')

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9, 0.95), weight_decay=0.01)
    ema_state = {n: t.detach().float().clone() for n, t in model.state_dict().items()}

    pos = 0
    t0 = time.time()
    losses = []

    for step in range(1, STEPS + 1):
        usable = (batch_bytes // seq_len) * seq_len
        end = pos + usable + 1
        if end > len(train_data): pos = 0; end = usable + 1
        chunk = train_data[pos:end]; pos = end - 1
        x = torch.tensor(chunk[:-1].reshape(-1, seq_len), device=device)
        y = torch.tensor(chunk[1:].reshape(-1, seq_len), device=device)

        optimizer.zero_grad()
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            loss = model(x, y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

        with torch.no_grad():
            for n, t in model.state_dict().items():
                ema_state[n].mul_(EMA_DECAY).add_(t.detach().float(), alpha=1.0 - EMA_DECAY)

        if step % LOG_EVERY == 0 or step == 1:
            avg = np.mean(losses[-LOG_EVERY:])
            elapsed = time.time() - t0
            eta = elapsed / step * (STEPS - step)
            print(f'  step {step:>5} | loss {avg:.4f} | {elapsed/60:.1f}min | ~{eta/60:.0f}min left')

    total_time = time.time() - t0

    # Validation with EMA
    orig = {n: t.clone() for n, t in model.state_dict().items()}
    model.load_state_dict({n: t.to(orig[n].dtype) for n, t in ema_state.items()}, strict=True)
    val_losses = []
    with torch.no_grad():
        for i in range(0, min(1000 * seq_len, len(val_data) - seq_len - 1), seq_len):
            chunk = val_data[i:i + seq_len + 1]
            x = torch.tensor(chunk[:-1].reshape(1, -1), device=device)
            y = torch.tensor(chunk[1:].reshape(1, -1), device=device)
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                val_losses.append(model(x, y).item())
    model.load_state_dict(orig, strict=True)

    val_loss = np.mean(val_losses)

    if is_byte_level:
        # Byte-level: BPB = loss / ln(2) directly
        val_bpb = val_loss / math.log(2)
    else:
        # Token-level: need to convert via bytes_per_token
        # BPB = bits_per_token * tokens_per_byte
        # bits_per_token = loss / ln(2)
        # tokens_per_byte = len(token_val) / len(byte_val)
        tokens_per_byte = len(token_val) / len(byte_val)
        val_bpb = (val_loss / math.log(2)) * tokens_per_byte

    print(f'  VAL: loss={val_loss:.4f} | BPB={val_bpb:.4f} | {total_time/60:.1f}min | {p/1e6:.1f}M params')
    return {'name': name, 'params': p, 'val_loss': val_loss, 'val_bpb': val_bpb, 'time': total_time}

print('Harness ready')

In [ ]:
# ============================================================
# RUN BOTH MODELS
# ============================================================
#
# Same architecture (9 layers, 512 dim, 8 heads, 4 KV, 3x MLP)
# Same optimizer (AdamW, lr=3e-4)
# Same steps (7000)
# Same GPU
# Same source data (2 train shards, 1 val shard)
#
# Only difference: vocab size (1024 tokens vs 260 bytes)
# Token model uses seq_len=1024 tokens (~2500 bytes of context)
# Byte model uses seq_len=1024 bytes (~1024 bytes of context)

results = []

# TOKEN-LEVEL (competition baseline architecture)
torch.manual_seed(1337)
results.append(train_and_val(
    'Token-level (1024 vocab)',
    LM(vocab_size=1024, dim=512, num_layers=9, nh=8, nkv=4, mlp_mult=3),
    token_train, token_val,
    seq_len=1024, batch_bytes=32768, is_byte_level=False,
))

# BYTE-LEVEL (same architecture, 260 vocab)
torch.manual_seed(1337)
results.append(train_and_val(
    'Byte-level (260 vocab)',
    LM(vocab_size=260, dim=512, num_layers=9, nh=8, nkv=4, mlp_mult=3),
    byte_train, byte_val,
    seq_len=1024, batch_bytes=32768, is_byte_level=True,
))

In [ ]:
# ============================================================
# RESULTS
# ============================================================

print()
print('='*70)
print('APPLES TO APPLES: TOKEN vs BYTE')
print('='*70)
print(f'  Same GPU: {gpu}')
print(f'  Same optimizer: AdamW, lr={LR}')
print(f'  Same steps: {STEPS}')
print(f'  Same source data: 2 train shards')
print()

for r in results:
    print(f'  {r["name"]}:')
    print(f'    Params:   {r["params"]:,} ({r["params"]/1e6:.1f}M)')
    print(f'    Val BPB:  {r["val_bpb"]:.4f}')
    print(f'    Val loss: {r["val_loss"]:.4f}')
    print(f'    Time:     {r["time"]/60:.1f} min')
    print()

t = results[0]  # token
b = results[1]  # byte
delta = b['val_bpb'] - t['val_bpb']
print(f'  Delta BPB: {delta:+.4f} (negative = byte wins)')
print()
if abs(delta) < 0.05:
    print('  >>> ROUGHLY EQUAL — byte-level matches token-level!')
    print('  >>> The 1.69 vs 1.22 gap was infrastructure, not architecture.')
elif delta < 0:
    print('  >>> BYTE-LEVEL WINS')
else:
    print(f'  >>> Token-level wins by {delta:.4f} BPB')
    if delta < 0.2:
        print('  >>> Gap is small — byte-level is viable with better training')
    else:
        print('  >>> Significant gap — byte-level has a structural disadvantage')

print()
print(f'  Competition baseline (8xH100, 80 shards, Muon): 1.2244 BPB')
print(f'  If token-level here is ~{t["val_bpb"]:.2f}, the infrastructure gap is ~{t["val_bpb"]-1.22:.2f} BPB')